In [0]:
# Databricks notebook source
# ══════════════════════════════════════
# SILVER — physical_itens_venda_caixa
# Squad 3 — Arquitetura Medalhao
# Regra: tratamentos e regras tecnicas
# Frequencia: toda segunda-feira as 5h
# ══════════════════════════════════════

# COMMAND ----------



In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# constantes do notebook

BRONZE_TABLE       = "physical_itens_venda_caixa"
BRONZE_PATH        = f"{BRONZE_BASE_PATH}{BRONZE_TABLE}"
BRONZE_VENDAS_PATH = f"{BRONZE_BASE_PATH}physical_vendas_caixa"

SILVER_TABLE = "physical_itens_venda_caixa"
SILVER_PATH  = f"{SILVER_BASE_PATH}{SILVER_TABLE}"

KEY_COLUMNS       = ["id_item_venda"]
SILVER_WRITE_MODE = "overwrite"

SILVER_REQUIRED_COLUMNS = [
    "id_item_venda",
    "id_transacao",
    "codigo_barras_produto",
    "quantidade",
    "preco_unitario_registro",
    "valor_total_item",
    "id_loja",
    "dt_venda",
    "tipo_pagamento",
    "ano",
    "mes",
    "bronze_ingested_at",
    "bronze_source_file",
    "silver_processed_at",
]

print("Constantes configuradas:")
print(f"   BRONZE_PATH        : {BRONZE_PATH}")
print(f"   BRONZE_VENDAS_PATH : {BRONZE_VENDAS_PATH}")
print(f"   SILVER_PATH        : {SILVER_PATH}")
print(f"   KEY_COLUMNS        : {KEY_COLUMNS}")

In [0]:
# configuracoes do ADLS

adls_options = get_adls_options()
print("Opcoes ADLS configuradas.")

In [0]:
# ler Bronze itens

df_bronze = read_delta(
    spark        = spark,
    path         = BRONZE_PATH,
    adls_options = adls_options
)

df_bronze.printSchema()
total_bronze = df_bronze.count()
print(f"Total de registros na Bronze: {total_bronze:,}")
display(df_bronze.limit(10))

In [0]:
# ler Bronze vendas para enriquecimento
# adiciona: id_loja, dt_venda, tipo_pagamento

df_vendas_bronze = read_delta(
    spark        = spark,
    path         = BRONZE_VENDAS_PATH,
    adls_options = adls_options
)

df_vendas_ref = df_vendas_bronze.select(
    "id_transacao",
    "id_loja",
    "dt_venda",
    "tipo_pagamento"
)

print(f"Total vendas referencia: {df_vendas_ref.count():,}")
display(df_vendas_ref.limit(5))

In [0]:
# Regra 2 — validar FK
# id_transacao deve existir em physical_vendas_caixa

ids_vendas = df_vendas_ref.select("id_transacao").distinct()
ids_itens  = df_bronze.select("id_transacao").distinct()

orfaos = ids_itens.subtract(ids_vendas).count()

print(f"Transacoes unicas em itens  : {ids_itens.count():,}")
print(f"Transacoes unicas em vendas : {ids_vendas.count():,}")
print(f"Itens orfaos (sem venda pai): {orfaos:,}")

if orfaos > 0:
    print(f"Atencao: {orfaos} id_transacao orfaos encontrados.")
else:
    print("Validacao OK: todos os id_transacao existem em vendas_caixa.")

In [0]:
# validar conversoes antes de aplicar

from pyspark.sql.functions import col, count, when

df_validacao_conv = df_bronze.select(
    count("*").alias("total_linhas"),
    count(
        when(
            col("id_item_venda").isNotNull() &
            col("id_item_venda").cast("long").isNull(),
            True
        )
    ).alias("falhas_id_item_venda"),
    count(
        when(
            col("quantidade").isNotNull() &
            col("quantidade").cast("double").isNull(),
            True
        )
    ).alias("falhas_quantidade"),
    count(
        when(
            col("preco_unitario_registro").isNotNull() &
            col("preco_unitario_registro").cast("double").isNull(),
            True
        )
    ).alias("falhas_preco"),
    count(
        when(
            col("valor_total_item").isNotNull() &
            col("valor_total_item").cast("double").isNull(),
            True
        )
    ).alias("falhas_valor_total"),
)

display(df_validacao_conv)
validacao_conv = df_validacao_conv.collect()[0]

if validacao_conv["falhas_quantidade"] > 0:
    raise Exception(
        f"Existem {validacao_conv['falhas_quantidade']} valores "
        f"de quantidade que nao podem ser convertidos para double."
    )
if validacao_conv["falhas_preco"] > 0:
    raise Exception(
        f"Existem {validacao_conv['falhas_preco']} valores "
        f"de preco_unitario_registro que nao podem ser convertidos."
    )
if validacao_conv["falhas_valor_total"] > 0:
    raise Exception(
        f"Existem {validacao_conv['falhas_valor_total']} valores "
        f"de valor_total_item que nao podem ser convertidos."
    )

print("Validacao OK: conversoes principais podem ser feitas.")

In [0]:
# aplicar transformacoes Silver

from pyspark.sql.functions import (
    col, trim, abs as spark_abs,
    current_timestamp,
    round as spark_round,
    row_number
)
from pyspark.sql.types import LongType, DoubleType
from pyspark.sql.window import Window

# deduplicacao por id_item_venda
window_dedup = (
    Window
    .partitionBy("id_item_venda")
    .orderBy(col("bronze_ingested_at").desc_nulls_last())
)

df_silver = (
    df_bronze

    # JOIN com vendas para enriquecimento
    .join(df_vendas_ref, on="id_transacao", how="left")

    # converter tipos
    .withColumn("id_item_venda",
        col("id_item_venda").cast(LongType()))
    .withColumn("quantidade",
        col("quantidade").cast(DoubleType()))
    .withColumn("preco_unitario_registro",
        col("preco_unitario_registro").cast(DoubleType()))
    .withColumn("valor_total_item",
        col("valor_total_item").cast(DoubleType()))

    # padronizar texto
    .withColumn("codigo_barras_produto",
        trim(col("codigo_barras_produto")))
    .withColumn("tipo_pagamento",
        trim(col("tipo_pagamento")))

    # deduplicacao
    .withColumn("rn", row_number().over(window_dedup))
    .filter(col("rn") == 1)
    .drop("rn")

    # metadado silver
    .withColumn("silver_processed_at", current_timestamp())
)

In [0]:
# Regra 1 — id_item_venda PK: sem nulos, sem duplicatas

resultado_pk = validate_key_columns(
    df          = df_silver,
    key_columns = KEY_COLUMNS
)
print(resultado_pk["message"])

In [0]:
# Regra 3 — quantidade deve ser > 0

qtd_invalida = df_silver.filter(
    col("quantidade").isNull() |
    (col("quantidade") <= 0)
).count()

if qtd_invalida > 0:
    print(f"Atencao: {qtd_invalida} registros com quantidade <= 0.")
    df_silver = df_silver.filter(col("quantidade") > 0)
    print(f"Registros removidos. Total Silver: {df_silver.count():,}")
else:
    print("Validacao OK: todos os registros tem quantidade > 0.")

In [0]:
# Regra 4 — preco_unitario_registro deve ser > 0

preco_invalido = df_silver.filter(
    col("preco_unitario_registro").isNull() |
    (col("preco_unitario_registro") <= 0)
).count()

if preco_invalido > 0:
    print(f"Atencao: {preco_invalido} registros com preco_unitario_registro <= 0.")
    df_silver.filter(
        col("preco_unitario_registro") <= 0
    ).select(
        "id_item_venda",
        "codigo_barras_produto",
        "preco_unitario_registro"
    ).show(10)
else:
    print("Validacao OK: todos os registros tem preco_unitario_registro > 0.")

In [0]:
# Regra 5 — valor_total_item = preco * quantidade
# tolerancia de R$ 0.05

df_silver = df_silver.withColumn(
    "valor_calculado",
    spark_round(
        col("preco_unitario_registro") * col("quantidade"),
        2
    )
)

inconsistencias = df_silver.filter(
    spark_abs(
        col("valor_calculado") - col("valor_total_item")
    ) > 0.05
).count()

if inconsistencias > 0:
    print(f"Atencao: {inconsistencias} registros com inconsistencia em valor_total_item.")
    df_silver.filter(
        spark_abs(
            col("valor_calculado") - col("valor_total_item")
        ) > 0.05
    ).select(
        "id_item_venda",
        "quantidade",
        "preco_unitario_registro",
        "valor_total_item",
        "valor_calculado"
    ).show(10)
else:
    print("Validacao OK: valor_total_item consistente com preco * quantidade.")

# remove coluna auxiliar
df_silver = df_silver.drop("valor_calculado")

In [0]:
# selecionar colunas finais da Silver

df_silver = df_silver.select(
    "id_item_venda",
    "id_transacao",
    "codigo_barras_produto",
    "quantidade",
    "preco_unitario_registro",
    "valor_total_item",
    "id_loja",
    "dt_venda",
    "tipo_pagamento",
    "ano",
    "mes",
    "bronze_ingested_at",
    "bronze_source_file",
    "silver_processed_at",
)

In [0]:
# visualizar Silver

df_silver.printSchema()
total_silver = df_silver.count()

print(f"Total Bronze : {total_bronze:,}")
print(f"Total Silver : {total_silver:,}")
display(df_silver.limit(10))

In [0]:
# validar qualidade Silver

validate_silver_quality(
    df               = df_silver,
    required_columns = SILVER_REQUIRED_COLUMNS
)

In [0]:
# gravar Silver Delta no ADLS
# particionado por ano e mes

write_delta(
    df           = df_silver,
    path         = SILVER_PATH,
    mode         = SILVER_WRITE_MODE,
    partition_by = ["ano", "mes"],
    adls_options = adls_options
)

In [0]:
# ler Silver gravada para validacao

df_silver_saved = read_delta(
    spark        = spark,
    path         = SILVER_PATH,
    adls_options = adls_options
)

df_silver_saved.printSchema()
display(df_silver_saved.limit(10))

In [0]:
# validar Silver gravada

compare_row_counts(
    source_df = df_silver,
    target_df = df_silver_saved,
    label     = "Silver memoria x Silver gravada"
)

In [0]:
# validar schema final da Silver

colunas_ausentes = [
    c for c in SILVER_REQUIRED_COLUMNS
    if c not in df_silver_saved.columns
]

if colunas_ausentes:
    raise Exception(
        f"Colunas obrigatorias ausentes na Silver: {colunas_ausentes}"
    )

print("Validacao schema final Silver OK.")

In [0]:
# validar particoes gravadas

display(
    df_silver_saved
    .groupBy("ano", "mes")
    .count()
    .orderBy("ano", "mes")
)

In [0]:
# validar qualidade final Silver gravada

validate_silver_quality(
    df               = df_silver_saved,
    required_columns = SILVER_REQUIRED_COLUMNS
)

In [0]:
# Resumo

print("=" * 55)
print("SILVER physical_itens_venda_caixa concluida!")
print("=" * 55)
print(f"""
   Bronze Path  : {BRONZE_PATH}
   Silver Path  : {SILVER_PATH}
   Total Bronze : {total_bronze:,}
   Total Silver : {total_silver:,}
   Particao     : ano e mes

Regras tecnicas aplicadas:
   Regra 1 : id_item_venda PK — sem nulos, sem duplicatas
   Regra 2 : id_transacao FK — validado vs vendas_caixa
   Regra 3 : quantidade > 0
   Regra 4 : preco_unitario_registro > 0
   Regra 5 : valor_total_item = preco * quantidade

   Status   : SUCESSO
""")